In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k.csv")

print("Original Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. CHECK TARGET VARIABLE
# ============================================================

print("\nMissing PlacementStatus values:")
print(df["PlacementStatus"].isna().sum())

print("\nPlacementStatus values:")
print(df["PlacementStatus"].value_counts(dropna=False))


# ============================================================
# 3. REMOVE MISSING TARGET ROW
# ============================================================

df = df.dropna(subset=["PlacementStatus"]).copy()

print("\nDataset Shape after removing missing target:")
print(df.shape)


# ============================================================
# 4. TARGET VARIABLE
# ============================================================

# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"].astype(int)


# ============================================================
# 5. SELECT FEATURES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

X = df[features].copy()


# ============================================================
# 6. CHECK DATA TYPES
# ============================================================

print("\n============================================")
print("FEATURE DATA TYPES")
print("============================================")

print(X.dtypes)


# ============================================================
# 7. AUTOMATICALLY IDENTIFY NUMERICAL FEATURES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()


# ============================================================
# 8. AUTOMATICALLY IDENTIFY CATEGORICAL FEATURES
# ============================================================

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


# ============================================================
# 9. NUMERICAL PREPROCESSING
# ============================================================

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 10. CATEGORICAL PREPROCESSING
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)


# ============================================================
# 11. COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ============================================================
# 12. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data :", X_test.shape)


# ============================================================
# 13. BINOMIAL LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ============================================================
# 14. TRAIN MODEL
# ============================================================

logistic_model.fit(
    X_train,
    y_train
)

print("\nModel training completed successfully.")


# ============================================================
# 15. PREDICTION
# ============================================================

y_pred = logistic_model.predict(X_test)

y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 16. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

cm = confusion_matrix(
    y_test,
    y_pred
)

report = classification_report(
    y_test,
    y_pred,
    target_names=[
        "Not Placed",
        "Placed"
    ]
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)


# ============================================================
# 17. DISPLAY RESULTS
# ============================================================

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(report)

print("\nROC-AUC Score:")
print(round(roc_auc, 4))

Original Dataset Shape: (22694, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']

Missing PlacementStatus values:
1

PlacementStatus values:
PlacementStatus
0.0    11902
1.0    10791
NaN        1
Name: count, dtype: int64

Dataset Shape after removing missing target:
(22693, 31)

FEATURE DATA TYPES
Gender                 object
City                   object
CollegeTier            object
Stream                 object
Specialisation         object
Hostel                 object
HistoryOfBacklogs      object
SGPA_Sem1             float64
SGPA_Sem2             float64
SGPA_S

L1 Logistic Regression

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k.csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. REMOVE MISSING TARGET VALUES
# ============================================================

# PlacementStatus:
# 0 = Not Placed
# 1 = Placed

df = df.dropna(subset=["PlacementStatus"]).copy()

y = df["PlacementStatus"].astype(int)


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

X = df[features].copy()


# ============================================================
# 4. IDENTIFY CATEGORICAL VARIABLES
# ============================================================

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


# ============================================================
# 5. IDENTIFY NUMERICAL VARIABLES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()


print("\nCategorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)


# ============================================================
# 6. NUMERICAL PREPROCESSING
# ============================================================

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 7. CATEGORICAL PREPROCESSING
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)


# ============================================================
# 8. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ============================================================
# 9. TRAIN-TEST SPLIT
# ============================================================

# stratify=y keeps the same proportion of
# Placed and Not Placed students in both datasets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data :", X_test.shape)


# ============================================================
# 10. BINOMIAL LOGISTIC REGRESSION - L1
# ============================================================

l1_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ============================================================
# 11. TRAIN MODEL
# ============================================================

l1_model.fit(
    X_train,
    y_train
)

print("\nL1 Logistic Regression model trained successfully.")


# ============================================================
# 12. PREDICTION
# ============================================================

y_pred_l1 = l1_model.predict(X_test)

# Probability of Placement = class 1
y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]


# ============================================================
# 13. MODEL EVALUATION
# ============================================================

accuracy_l1 = accuracy_score(
    y_test,
    y_pred_l1
)

confusion_l1 = confusion_matrix(
    y_test,
    y_pred_l1
)

classification_l1 = classification_report(
    y_test,
    y_pred_l1,
    target_names=[
        "Not Placed",
        "Placed"
    ]
)

roc_auc_l1 = roc_auc_score(
    y_test,
    y_prob_l1
)


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION - L1 RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy_l1, 4))

print("\nConfusion Matrix:")
print(confusion_l1)

print("\nClassification Report:")
print(classification_l1)

print("\nROC-AUC:")
print(round(roc_auc_l1, 4))

Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']

Categorical Features:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'ExtraCurricular']

Numerical Features:
['SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore']

Training Data: (40000, 27)
Testing Data : (10000, 27)

L1

L2 REGRESSION

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k.csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. REMOVE MISSING TARGET VALUES
# ============================================================

# PlacementStatus:
# 0 = Not Placed
# 1 = Placed

df = df.dropna(subset=["PlacementStatus"]).copy()

y = df["PlacementStatus"].astype(int)


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

X = df[features].copy()


# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()


print("\nCategorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)


# ============================================================
# 6. NUMERICAL PREPROCESSING
# ============================================================

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 7. CATEGORICAL PREPROCESSING
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)


# ============================================================
# 8. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ============================================================
# 9. TRAIN-TEST SPLIT
# ============================================================

# stratify=y keeps the same proportion of
# Placed and Not Placed students in both datasets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data :", X_test.shape)


# ============================================================
# 10. BINOMIAL LOGISTIC REGRESSION - L2
# ============================================================

l2_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ============================================================
# 11. TRAIN MODEL
# ============================================================

l2_model.fit(
    X_train,
    y_train
)

print("\nL2 Logistic Regression model trained successfully.")


# ============================================================
# 12. PREDICTION
# ============================================================

y_pred_l2 = l2_model.predict(X_test)

# Probability of Placement = class 1
y_prob_l2 = l2_model.predict_proba(X_test)[:, 1]


# ============================================================
# 13. MODEL EVALUATION
# ============================================================

accuracy_l2 = accuracy_score(
    y_test,
    y_pred_l2
)

confusion_l2 = confusion_matrix(
    y_test,
    y_pred_l2
)

classification_l2 = classification_report(
    y_test,
    y_pred_l2,
    target_names=[
        "Not Placed",
        "Placed"
    ]
)

roc_auc_l2 = roc_auc_score(
    y_test,
    y_prob_l2
)


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION - L2 RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy_l2, 4))

print("\nConfusion Matrix:")
print(confusion_l2)

print("\nClassification Report:")
print(classification_l2)

print("\nROC-AUC:")
print(round(roc_auc_l2, 4))

Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']

Categorical Features:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'ExtraCurricular']

Numerical Features:
['SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore']

Training Data: (40000, 27)
Testing Data : (10000, 27)

L2